# Mini projekt: Analiza razvoja država kroz vrijeme

## Tema

U ovom primjeru koristimo javni **Gapminder** dataset koji sadrži podatke o državama kroz vrijeme, uključujući:

- očekivani životni vijek (`lifeExp`)
- broj stanovnika (`pop`)
- BDP po stanovniku (`gdpPercap`)
- kontinent (`continent`)
- godinu (`year`)

## Glavno pitanje

**Postoji li povezanost između BDP-a po stanovniku i očekivanog životnog vijeka te kako se ti pokazatelji mijenjaju kroz vrijeme?**

Kroz primjer ćemo proći tipičan analitički tijek:

1. učitavanje podataka
2. pregled strukture podataka
3. osnovno čišćenje i prilagodba
4. analiza pomoću Pandasa
5. vizualizacija pomoću `plot()`
6. dodatna obrada pomoću NumPyja
7. jednostavan zaključak na temelju podataka


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 1. Učitavanje podataka

Dataset ćemo učitati iz javno dostupnog CSV izvora.

Ako radite bez interneta, datoteku možete prethodno preuzeti i zatim u `read_csv()` upisati lokalnu putanju.


In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv"

df = pd.read_csv(url)
df.head()


## 2. Pregled strukture podataka

Prvo želimo razumjeti:

- koliko redaka i stupaca imamo
- kako se zovu stupci
- koje tipove podataka imamo
- ima li nedostajućih vrijednosti


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

## 3. Prilagodba i osnovno čišćenje podataka

U ovom datasetu nazivi stupaca nisu loši, ali ih možemo učiniti malo čitljivijima.

Također ćemo provjeriti postoje li nelogične vrijednosti, primjerice:

- negativan broj stanovnika
- negativan BDP po stanovniku
- očekivani životni vijek manji ili jednak nuli


In [ ]:
df = df.rename(columns={
    "country": "drzava",
    "continent": "kontinent",
    "year": "godina",
    "lifeExp": "zivotni_vijek",
    "pop": "stanovnistvo",
    "gdpPercap": "bdp_po_stanovniku"
})

df.head()


In [ ]:
nelogicne_vrijednosti = df[
    (df["stanovnistvo"] <= 0) |
    (df["bdp_po_stanovniku"] <= 0) |
    (df["zivotni_vijek"] <= 0)
]

nelogicne_vrijednosti


U ovom slučaju nemamo problematičnih vrijednosti koje bismo morali ukloniti.

Ipak, u stvarnim projektima ovaj je korak važan jer pogrešni ili nelogični podaci mogu značajno promijeniti rezultat analize.


## 4. Osnovna pitanja o datasetu

Pogledajmo koje godine i kontinente dataset pokriva.


In [ ]:
df["godina"].min(), df["godina"].max()

In [ ]:
df["kontinent"].unique()

In [ ]:
df["drzava"].nunique()

## 5. Prosječni životni vijek kroz vrijeme

Prvo ćemo izračunati prosječni životni vijek po godini.

Ovdje koristimo `groupby()` jer želimo grupirati podatke po godini.


In [ ]:
zivotni_vijek_po_godini = (
    df
    .groupby("godina")["zivotni_vijek"]
    .mean()
)

zivotni_vijek_po_godini


In [ ]:
zivotni_vijek_po_godini.plot(
    kind="line",
    marker="o",
    figsize=(10, 5),
    title="Prosječni životni vijek kroz vrijeme"
)

plt.xlabel("Godina")
plt.ylabel("Prosječni životni vijek")
plt.grid(True)
plt.show()


## 6. Prosječni BDP po stanovniku kroz vrijeme

Sada ćemo napraviti sličnu analizu za BDP po stanovniku.


In [ ]:
bdp_po_godini = (
    df
    .groupby("godina")["bdp_po_stanovniku"]
    .mean()
)

bdp_po_godini


In [ ]:
bdp_po_godini.plot(
    kind="line",
    marker="o",
    figsize=(10, 5),
    title="Prosječni BDP po stanovniku kroz vrijeme"
)

plt.xlabel("Godina")
plt.ylabel("Prosječni BDP po stanovniku")
plt.grid(True)
plt.show()


## 7. Usporedba kontinenata

Sada želimo vidjeti postoje li razlike između kontinenata.

Izračunat ćemo prosječni životni vijek i prosječni BDP po stanovniku po kontinentu.


In [ ]:
analiza_po_kontinentu = (
    df
    .groupby("kontinent")
    .agg(
        prosjecni_zivotni_vijek=("zivotni_vijek", "mean"),
        prosjecni_bdp_po_stanovniku=("bdp_po_stanovniku", "mean"),
        prosjecno_stanovnistvo=("stanovnistvo", "mean"),
        broj_drzava=("drzava", "nunique")
    )
    .sort_values("prosjecni_zivotni_vijek", ascending=False)
)

analiza_po_kontinentu


In [ ]:
analiza_po_kontinentu["prosjecni_zivotni_vijek"].plot(
    kind="bar",
    figsize=(9, 5),
    title="Prosječni životni vijek po kontinentu"
)

plt.xlabel("Kontinent")
plt.ylabel("Prosječni životni vijek")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()


In [ ]:
analiza_po_kontinentu["prosjecni_bdp_po_stanovniku"].plot(
    kind="bar",
    figsize=(9, 5),
    title="Prosječni BDP po stanovniku po kontinentu"
)

plt.xlabel("Kontinent")
plt.ylabel("Prosječni BDP po stanovniku")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()


## 8. Analiza za zadnju dostupnu godinu

Za jasniju usporedbu uzet ćemo samo zadnju godinu u datasetu.

To nam omogućuje da uspoređujemo države u istoj vremenskoj točki.


In [ ]:
zadnja_godina = df["godina"].max()
zadnja_godina

In [ ]:
df_zadnja = df[df["godina"] == zadnja_godina].copy()
df_zadnja.head()


In [ ]:
df_zadnja.sort_values("zivotni_vijek", ascending=False).head(10)


In [ ]:
df_zadnja.sort_values("bdp_po_stanovniku", ascending=False).head(10)


## 9. Veza između BDP-a po stanovniku i životnog vijeka

Sada želimo odgovoriti na glavno pitanje:

**Imaju li države s većim BDP-om po stanovniku u prosjeku i veći očekivani životni vijek?**

Za početak ćemo napraviti scatter plot.


In [ ]:
df_zadnja.plot(
    kind="scatter",
    x="bdp_po_stanovniku",
    y="zivotni_vijek",
    figsize=(9, 6),
    title=f"BDP po stanovniku i životni vijek u {zadnja_godina}. godini"
)

plt.xlabel("BDP po stanovniku")
plt.ylabel("Životni vijek")
plt.grid(True)
plt.show()


Na ovom grafu često se vidi da odnos nije potpuno linearan.

Razlike između država s malim BDP-om mogu biti velike, dok se kod bogatijih država životni vijek više stabilizira.

Zato ćemo dodati i logaritamsku transformaciju BDP-a pomoću NumPyja.


In [ ]:
df_zadnja["log_bdp_po_stanovniku"] = np.log(df_zadnja["bdp_po_stanovniku"])

df_zadnja[["drzava", "bdp_po_stanovniku", "log_bdp_po_stanovniku", "zivotni_vijek"]].head()


In [ ]:
df_zadnja.plot(
    kind="scatter",
    x="log_bdp_po_stanovniku",
    y="zivotni_vijek",
    figsize=(9, 6),
    title=f"Log BDP-a po stanovniku i životni vijek u {zadnja_godina}. godini"
)

plt.xlabel("Log BDP-a po stanovniku")
plt.ylabel("Životni vijek")
plt.grid(True)
plt.show()


## 10. Korelacija pomoću Pandasa i NumPyja

Korelacija nam govori koliko su dvije numeričke varijable povezane.

Vrijednost korelacije može biti:

- blizu `1`: jaka pozitivna veza
- blizu `0`: slaba ili nikakva linearna veza
- blizu `-1`: jaka negativna veza


In [ ]:
korelacija_pandas = df_zadnja["bdp_po_stanovniku"].corr(df_zadnja["zivotni_vijek"])
korelacija_pandas


In [ ]:
korelacija_log_pandas = df_zadnja["log_bdp_po_stanovniku"].corr(df_zadnja["zivotni_vijek"])
korelacija_log_pandas


In [ ]:
x = df_zadnja["log_bdp_po_stanovniku"].to_numpy()
y = df_zadnja["zivotni_vijek"].to_numpy()

korelacija_numpy = np.corrcoef(x, y)[0, 1]
korelacija_numpy


## 11. Standardizacija podataka pomoću NumPyja

Standardizacija znači da vrijednosti pretvaramo tako da imaju:

- prosjek približno `0`
- standardnu devijaciju približno `1`

To je korisno kada želimo uspoređivati varijable koje su na različitim skalama.


In [ ]:
vrijednosti = df_zadnja[["log_bdp_po_stanovniku", "zivotni_vijek"]].to_numpy()

prosjeci = vrijednosti.mean(axis=0)
standardne_devijacije = vrijednosti.std(axis=0)

standardizirane_vrijednosti = (vrijednosti - prosjeci) / standardne_devijacije

standardizirane_vrijednosti[:5]


In [ ]:
df_zadnja["standardizirani_log_bdp"] = standardizirane_vrijednosti[:, 0]
df_zadnja["standardizirani_zivotni_vijek"] = standardizirane_vrijednosti[:, 1]

df_zadnja[[
    "drzava",
    "kontinent",
    "standardizirani_log_bdp",
    "standardizirani_zivotni_vijek"
]].head()


## 12. Jednostavni indeks razvoja

Sada ćemo napraviti vrlo jednostavan indeks koji kombinira:

- standardizirani log BDP-a po stanovniku
- standardizirani životni vijek

Ovo nije službeni indeks i ne smije se koristiti kao ozbiljan ekonomski pokazatelj.

Služi samo kao vježba kako kombinirati Pandas i NumPy.


In [ ]:
df_zadnja["jednostavni_indeks"] = (
    df_zadnja["standardizirani_log_bdp"] +
    df_zadnja["standardizirani_zivotni_vijek"]
) / 2

df_zadnja[[
    "drzava",
    "kontinent",
    "bdp_po_stanovniku",
    "zivotni_vijek",
    "jednostavni_indeks"
]].sort_values("jednostavni_indeks", ascending=False).head(10)


In [ ]:
top_10 = (
    df_zadnja
    .sort_values("jednostavni_indeks", ascending=False)
    .head(10)
    .set_index("drzava")
)

top_10["jednostavni_indeks"].plot(
    kind="bar",
    figsize=(10, 5),
    title="Top 10 država prema jednostavnom indeksu"
)

plt.xlabel("Država")
plt.ylabel("Jednostavni indeks")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()


## 13. Zaključak

Na temelju ove analize možemo zaključiti:

1. Prosječni životni vijek kroz vrijeme raste.
2. Prosječni BDP po stanovniku također raste, ali razlike između kontinenata i država ostaju značajne.
3. Postoji pozitivna veza između BDP-a po stanovniku i očekivanog životnog vijeka.
4. Logaritamska transformacija BDP-a često bolje prikazuje odnos između ekonomskog razvoja i životnog vijeka.
5. NumPy nam je koristan za matematičke transformacije, korelaciju i standardizaciju podataka.


## 14. Dodatni zadaci za polaznike

1. Pronađite 10 država s najnižim životnim vijekom u zadnjoj dostupnoj godini.
2. Izračunajte prosječni BDP po stanovniku samo za Europu kroz godine.
3. Nacrtajte linijski graf životnog vijeka za Hrvatsku, Njemačku i Japan.
4. Izračunajte koliko se životni vijek Hrvatske promijenio od prve do zadnje dostupne godine.
5. Napravite novi jednostavni indeks koji uključuje i broj stanovnika.
6. Usporedite korelaciju između BDP-a i životnog vijeka za svaki kontinent zasebno.


## 15. Primjer dodatne analize: Hrvatska

Ovaj dio pokazuje kako se može izdvojiti jedna država i analizirati njezin razvoj kroz vrijeme.


In [ ]:
hrvatska = df[df["drzava"] == "Croatia"].copy()
hrvatska


In [ ]:
hrvatska.plot(
    kind="line",
    x="godina",
    y="zivotni_vijek",
    marker="o",
    figsize=(10, 5),
    title="Životni vijek u Hrvatskoj kroz vrijeme"
)

plt.xlabel("Godina")
plt.ylabel("Životni vijek")
plt.grid(True)
plt.show()


In [ ]:
promjena_zivotnog_vijeka = (
    hrvatska["zivotni_vijek"].iloc[-1] -
    hrvatska["zivotni_vijek"].iloc[0]
)

promjena_zivotnog_vijeka


Ovaj rezultat pokazuje za koliko se godina povećao očekivani životni vijek u Hrvatskoj između prve i zadnje godine dostupne u datasetu.
